# Stage 9: reranking depth after fine-tuning

Compares depth 20 and 50 for the off-the-shelf and fine-tuned cross-encoders. Stage 5 measured lower recall at depth 50 with the off-the-shelf model, despite higher candidate-pool recall (0.9754 at depth 20 versus 0.9951 at depth 50 for Transformer 15/1).

Each query-chunk pair is scored once at depth 50; the depth-20 arm uses the corresponding subset. Its results must reproduce `stage8/final`.

Requires a T4 or comparable GPU, the fine-tuned checkpoint at `artifacts/models/bge_reranker_ft/final`, and the Stage 6 bench cache. Completed configurations are checkpointed.


## Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys
# Set RAG_PROJECT_DIR to the project root before running this cell.
PROJECT_DIR = os.environ.get('RAG_PROJECT_DIR')
if not PROJECT_DIR:
    raise RuntimeError('Set RAG_PROJECT_DIR to the project directory before running this notebook.')
os.environ['RAG_DATA_ROOT'] = PROJECT_DIR + '/artifacts'
sys.path.insert(0, PROJECT_DIR)
os.chdir(PROJECT_DIR)

import config as C
C.ensure_dirs()
print(C.summary())

## Install dependencies

In [ ]:
!pip install -q -r requirements.txt

## Smoke run: 30 questions

Checks the small evaluation path. It skips the Stage 8 reproduction check and writes outputs with a `smoke_` prefix. Passing this check does not validate the full evaluation.


In [ ]:
!python scripts/22_rerank_depth.py --max-questions 30

## Full evaluation

Resumes from completed configurations after interruption. The depth-20 arms must pass the `stage8/final` reproduction check before the depth comparison is interpreted.


In [ ]:
!python scripts/22_rerank_depth.py

## Review

In [ ]:
from IPython.display import Image, Markdown, display
import pathlib, os
latest = pathlib.Path(os.environ['RAG_DATA_ROOT']) / 'results' / 'latest'
display(Markdown((latest / 'stage9_depth_summary.md').read_text(encoding='utf-8')))
display(Image(str(latest / 'stage9_depth_delta.png')))

## Archive (only if the check passed)

In [ ]:
!python scripts/save_stage_results.py --stage stage9

## Depth comparison

The console compares fine-tuned depth-50 minus depth-20 Recall@1 with a 2 SE band:

- **DEEPER WINS:** positive difference exceeding the band.
- **NO DIFFERENCE:** difference within the band; this does not establish equivalence.
- **DEEPER LOSES:** negative difference exceeding the band.

The `ots` column provides the corresponding off-the-shelf comparison. Conclusions apply to the evaluated rerankers, configurations, and dataset.
